# AeroTrack — Part 5: Evaluate OSNet Re-Identification on VisDrone-MOT

## Objective

Use the ground-truth annotations supplied with VisDrone2019-MOT validation to evaluate whether pretrained OSNet can recognize the same person across different frames of a drone sequence.

This notebook does **not** use YOLO detections or BoT-SORT IDs to create labels. Instead, it uses the dataset's ground-truth boxes and target IDs. That lets us evaluate OSNet independently of our detector and tracker.

## Your downloaded folder

The notebook expects the folder structure you showed:

```text
Downloads/
└── VisDrone2019-MOT-val/
    └── VisDrone2019-MOT-val/
        ├── sequences/
        │   ├── uav0000086_00000_v/
        │   ├── uav0000117_02622_v/
        │   └── ...
        └── annotations/
            ├── uav0000086_00000_v.txt
            ├── uav0000117_02622_v.txt
            └── ...
```

## Annotation format

Each MOT ground-truth line contains ten comma-separated fields:

`frame_index, target_id, bbox_left, bbox_top, bbox_width, bbox_height, score, object_category, truncation, occlusion`

For this notebook:
- `frame_index`: sequence-local frame number, starting at 1.
- `target_id`: ground-truth identity within that sequence. ID 0 is valid and must not be discarded.
- `score`: 1 means use this ground-truth annotation; 0 means ignore it.
- `object_category`: 1 = pedestrian, 2 = people. We use both as humans.
- `truncation` and `occlusion`: attributes describing how much of the object is cut off or hidden.

Important: target IDs are only used within their own video sequence. We do not assume that target ID 12 in one sequence is the same person as target ID 12 in another sequence.

## Evaluation design

For each `(sequence, target_id)`:
1. Find valid person annotations.
2. Split that person's observations into an earlier and a later temporal half.
3. Select a clear crop from each half, with a minimum frame gap.
4. Extract OSNet embeddings for the earlier crop (gallery) and later crop (query).
5. Search the gallery of the same sequence for the matching ground-truth identity.

We report Rank-1 accuracy and mean Average Precision (mAP), plus pairwise similarity diagnostics. This is a within-sequence, temporal ReID evaluation. It is not cross-sequence identity evaluation.

The pretrained OSNet checkpoint is trained on general pedestrian imagery (MSMT17), not aerial imagery. Treat this as a baseline, not an expected production result.

## Section 1 — Imports and dataset paths

Update `VISDRONE_ROOT` only if your extracted dataset is in a different location.

In [ ]:
from pathlib import Path
import re
import math
import time
import json

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch

from IPython.display import display
from PIL import Image

# Your extracted VisDrone-MOT validation folder.
VISDRONE_ROOT = (
    Path.home()
    / "Downloads"
    / "VisDrone2019-MOT-val"
    / "VisDrone2019-MOT-val"
)

SEQUENCES_DIR = VISDRONE_ROOT / "sequences"
ANNOTATIONS_DIR = VISDRONE_ROOT / "annotations"

PROJECT_DIR = Path.cwd()
OUTPUT_DIR = PROJECT_DIR / "outputs" / "part5_visdrone_reid"
CROP_DIR = OUTPUT_DIR / "crops"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
CROP_DIR.mkdir(parents=True, exist_ok=True)

print("VisDrone root:", VISDRONE_ROOT)
print("Root exists:", VISDRONE_ROOT.exists())
print("Sequences exists:", SEQUENCES_DIR.exists())
print("Annotations exists:", ANNOTATIONS_DIR.exists())
print("Output folder:", OUTPUT_DIR)

## Section 2 — Check GPU and load the ReID package

We use the same pretrained OSNet model used in Part 3. This notebook does not train OSNet; it evaluates the pretrained model using labelled VisDrone person crops.

If you already installed `reid` in Part 3, you can skip the installation cell.

In [ ]:
if torch.cuda.is_available():
    REID_DEVICE = "cuda"
    print("GPU:", torch.cuda.get_device_name(0))
    print("CUDA:", torch.version.cuda)
else:
    REID_DEVICE = "cpu"
    print("CUDA is not available; using CPU.")

print("ReID device:", REID_DEVICE)

In [ ]:
%pip install reid scikit-learn

In [ ]:
from reid import ReIDModel
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    precision_recall_curve,
    auc
)

print("ReID and scikit-learn imports successful.")

## Section 3 — Configure the experiment

The default filters remove ignored annotations, non-person categories, extremely small boxes and heavily occluded boxes.

We include both VisDrone person categories:
- 1: pedestrian
- 2: people

We retain `target_id=0` because it is a valid ground-truth identity value in this dataset. We only exclude negative IDs.

The minimum crop dimensions are deliberately modest for overhead footage. We will inspect the crop-size distribution before interpreting the ReID metrics.

In [ ]:
# VisDrone MOT annotation columns.
ANNOTATION_COLUMNS = [
    "frame_index",
    "target_id",
    "bbox_left",
    "bbox_top",
    "bbox_width",
    "bbox_height",
    "score",
    "object_category",
    "truncation",
    "occlusion"
]

PERSON_CATEGORY_IDS = [1, 2]

# Crop filtering.
MIN_BOX_WIDTH = 12
MIN_BOX_HEIGHT = 20
MAX_OCCLUSION = 1       # 0 = none, 1 = partial; exclude heavy occlusion (2)
MAX_TRUNCATION = 1      # Keep none/partial truncation
MIN_OBSERVATIONS_PER_ID = 4
MIN_QUERY_GALLERY_GAP_FRAMES = 15

# ReID model settings.
REID_MODEL_NAME = "osnet_x1_0_msmt17_combineall"
EMBEDDING_BATCH_SIZE = 16
CROP_MARGIN_RATIO = 0.04

print("Person categories:", PERSON_CATEGORY_IDS)
print("Minimum frame gap:", MIN_QUERY_GALLERY_GAP_FRAMES)

## Section 4 — Discover sequences and match annotation files

Find the JPG frames in each sequence and match each sequence to an annotation text file with the same sequence name.

The image frame number is read from the trailing digits in the image filename. If filenames do not contain a numeric suffix, the notebook will stop and show the unexpected filename so we can adapt the mapping safely.

In [ ]:
if not SEQUENCES_DIR.exists() or not ANNOTATIONS_DIR.exists():
    raise FileNotFoundError(
        "Could not find the sequences/annotations folders. "
        "Check VISDRONE_ROOT in Section 1."
    )

sequence_dirs = sorted(
    path for path in SEQUENCES_DIR.iterdir()
    if path.is_dir()
)

if not sequence_dirs:
    raise RuntimeError("No sequence folders were found.")

sequence_data = {}

for sequence_dir in sequence_dirs:
    sequence_name = sequence_dir.name
    annotation_path = ANNOTATIONS_DIR / f"{sequence_name}.txt"

    if not annotation_path.exists():
        raise FileNotFoundError(
            f"Annotation file missing for {sequence_name}: {annotation_path}"
        )

    image_paths = sorted(
        [
            path for path in sequence_dir.rglob("*")
            if path.suffix.lower() in {".jpg", ".jpeg", ".png"}
        ],
        key=lambda p: p.as_posix()
    )

    if not image_paths:
        print("No image frames found:", sequence_name)
        continue

    frame_map = {}

    for image_path in image_paths:
        match = re.search(r"(\d+)$", image_path.stem)

        if match is None:
            raise ValueError(
                f"Cannot determine frame number from filename: {image_path.name}"
            )

        frame_number = int(match.group(1))

        if frame_number in frame_map:
            raise ValueError(
                f"Duplicate frame number {frame_number} in {sequence_name}"
            )

        frame_map[frame_number] = image_path

    sequence_data[sequence_name] = {
        "sequence_dir": sequence_dir,
        "annotation_path": annotation_path,
        "frame_map": frame_map,
        "frame_count": len(frame_map)
    }

    print(
        f"{sequence_name}: {len(frame_map)} frames | "
        f"frame numbers {min(frame_map)} to {max(frame_map)}"
    )

print("\nSequences discovered:", len(sequence_data))

## Section 5 — Read and validate MOT ground-truth annotations

Read each annotation file as ten comma-separated columns.

We validate the format and check that annotation frame numbers exist in the corresponding image sequence. We do not interpret a target ID as global across sequences.

In [ ]:
all_annotations = []

for sequence_name, info in sequence_data.items():
    annotation_path = info["annotation_path"]

    annotation_df = pd.read_csv(
        annotation_path,
        header=None,
        names=ANNOTATION_COLUMNS
    )

    if annotation_df.shape[1] != 10:
        raise ValueError(
            f"{sequence_name}: expected 10 annotation columns, "
            f"received {annotation_df.shape[1]}"
        )

    for column in ANNOTATION_COLUMNS:
        annotation_df[column] = pd.to_numeric(
            annotation_df[column], errors="coerce"
        )

    annotation_df = annotation_df.dropna(
        subset=ANNOTATION_COLUMNS
    ).copy()

    annotation_df["frame_index"] = annotation_df["frame_index"].astype(int)
    annotation_df["target_id"] = annotation_df["target_id"].astype(int)
    annotation_df["object_category"] = annotation_df["object_category"].astype(int)
    annotation_df["score"] = annotation_df["score"].astype(int)
    annotation_df["truncation"] = annotation_df["truncation"].astype(int)
    annotation_df["occlusion"] = annotation_df["occlusion"].astype(int)

    annotation_df["sequence_id"] = sequence_name

    available_frames = set(info["frame_map"].keys())
    missing_frame_numbers = sorted(
        set(annotation_df["frame_index"]) - available_frames
    )

    if missing_frame_numbers:
        print(
            f"WARNING {sequence_name}: "
            f"{len(missing_frame_numbers)} annotation frame numbers "
            "do not have a matching image."
        )

    all_annotations.append(annotation_df)

annotations = pd.concat(all_annotations, ignore_index=True)

print("Total annotation rows:", len(annotations))
print("Unique sequence-local target IDs:",
      annotations.groupby("sequence_id")["target_id"].nunique().sum())

print("\nObject category counts:")
display(
    annotations["object_category"]
    .value_counts()
    .sort_index()
    .rename_axis("category_id")
    .reset_index(name="rows")
)

print("\nGround-truth score counts:")
display(
    annotations["score"]
    .value_counts()
    .sort_index()
    .rename_axis("score")
    .reset_index(name="rows")
)

print("\nExample annotation rows:")
display(annotations.head())

## Section 6 — Visualize one ground-truth frame

Before extracting crops, draw the annotated person boxes on one source frame. This checks:
- The image-frame mapping is correct.
- The ten annotation fields were parsed correctly.
- The bounding boxes are in the expected top-left `x, y, width, height` format.
- Person classes 1 and 2 are selected.

In [ ]:
# Select the first sequence and the first frame containing a valid person label.
sample_sequence = None
sample_frame_number = None

valid_preview = annotations[
    (annotations["score"] == 1) &
    (annotations["object_category"].isin(PERSON_CATEGORY_IDS)) &
    (annotations["target_id"] >= 0)
]

for _, row in valid_preview.iterrows():
    sequence_name = row["sequence_id"]
    frame_number = int(row["frame_index"])

    if frame_number in sequence_data[sequence_name]["frame_map"]:
        sample_sequence = sequence_name
        sample_frame_number = frame_number
        break

if sample_sequence is None:
    raise RuntimeError("No valid person annotation could be matched to an image.")

sample_path = sequence_data[sample_sequence]["frame_map"][sample_frame_number]
sample_image = cv2.imread(str(sample_path))

if sample_image is None:
    raise RuntimeError(f"Could not read image: {sample_path}")

preview_annotations = valid_preview[
    (valid_preview["sequence_id"] == sample_sequence) &
    (valid_preview["frame_index"] == sample_frame_number)
]

preview_annotated = sample_image.copy()

for _, row in preview_annotations.iterrows():
    x = int(row["bbox_left"])
    y = int(row["bbox_top"])
    w = int(row["bbox_width"])
    h = int(row["bbox_height"])

    x1 = max(0, x)
    y1 = max(0, y)
    x2 = min(preview_annotated.shape[1], x + w)
    y2 = min(preview_annotated.shape[0], y + h)

    cv2.rectangle(
        preview_annotated,
        (x1, y1),
        (x2, y2),
        (0, 255, 0),
        2
    )

    label = f"GT ID {int(row['target_id'])} | class {int(row['object_category'])}"

    cv2.putText(
        preview_annotated,
        label,
        (x1, max(18, y1 - 5)),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.45,
        (0, 255, 0),
        1
    )

print("Sequence:", sample_sequence)
print("Frame:", sample_frame_number)
print("Image:", sample_path)
print("Person boxes in this frame:", len(preview_annotations))

plt.figure(figsize=(15, 9))
plt.imshow(cv2.cvtColor(preview_annotated, cv2.COLOR_BGR2RGB))
plt.title("VisDrone MOT ground-truth person annotations")
plt.axis("off")
plt.show()

## Section 7 — Filter valid person annotations

For this initial OSNet test, we keep:
- Ground-truth score equal to 1.
- Person categories 1 or 2.
- Non-negative target IDs (including ID 0).
- Boxes at least the configured minimum width and height.
- Occlusion and truncation within the configured limits.

This focuses the first evaluation on crops that have a reasonable chance of containing visible appearance information. Later, we can evaluate heavily occluded and smaller people separately.

In [ ]:
person_annotations = annotations[
    (annotations["score"] == 1) &
    (annotations["object_category"].isin(PERSON_CATEGORY_IDS)) &
    (annotations["target_id"] >= 0) &
    (annotations["bbox_width"] >= MIN_BOX_WIDTH) &
    (annotations["bbox_height"] >= MIN_BOX_HEIGHT) &
    (annotations["occlusion"] <= MAX_OCCLUSION) &
    (annotations["truncation"] <= MAX_TRUNCATION)
].copy()

person_annotations["identity_key"] = (
    person_annotations["sequence_id"].astype(str) +
    "::target_" +
    person_annotations["target_id"].astype(str)
)

person_annotations["box_area"] = (
    person_annotations["bbox_width"] *
    person_annotations["bbox_height"]
)

# Prefer larger, less-occluded boxes for representative crops.
person_annotations["crop_quality"] = (
    np.sqrt(person_annotations["box_area"].clip(lower=0)) *
    (1.0 - 0.20 * person_annotations["occlusion"]) *
    (1.0 - 0.10 * person_annotations["truncation"])
)

print("All annotation rows:", len(annotations))
print("Valid person rows after filtering:", len(person_annotations))
print("Person identities after filtering:",
      person_annotations["identity_key"].nunique())

print("\nPerson category counts after filtering:")
display(
    person_annotations.groupby("object_category")
    .size()
    .rename("rows")
    .reset_index()
)

print("\nBounding-box dimensions:")
display(
    person_annotations[["bbox_width", "bbox_height"]]
    .describe(percentiles=[0.25, 0.5, 0.75, 0.9])
)

## Section 8 — Create temporally separated gallery/query crops

We create one gallery crop from the earlier part of each person's track and one query crop from the later part.

This avoids evaluating the model on two adjacent, nearly identical frames of the same person.

For each sequence-local identity:
- Find its first and last annotated frame.
- Split its observed time span at the midpoint.
- Select the best-quality valid crop from the earlier half.
- Select the best-quality valid crop from the later half.
- Require at least `MIN_QUERY_GALLERY_GAP_FRAMES` between the two selected frames.

A person who does not have valid observations in both halves is not included in this initial temporal ReID test. That is an evaluation-sample limitation, not a detection failure.

In [ ]:
pair_records = []

for identity_key, group in person_annotations.groupby("identity_key"):
    group = group.sort_values("frame_index").copy()

    if len(group) < MIN_OBSERVATIONS_PER_ID:
        continue

    first_frame = int(group["frame_index"].min())
    last_frame = int(group["frame_index"].max())

    if last_frame <= first_frame:
        continue

    midpoint = (first_frame + last_frame) / 2.0

    early_group = group[group["frame_index"] <= midpoint]
    late_group = group[group["frame_index"] > midpoint]

    if early_group.empty or late_group.empty:
        continue

    # Find the most useful early and late crops.
    early_best = early_group.loc[
        early_group["crop_quality"].idxmax()
    ]
    late_best = late_group.loc[
        late_group["crop_quality"].idxmax()
    ]

    frame_gap = abs(
        int(late_best["frame_index"]) -
        int(early_best["frame_index"])
    )

    if frame_gap < MIN_QUERY_GALLERY_GAP_FRAMES:
        continue

    pair_records.append({
        "identity_key": identity_key,
        "sequence_id": str(early_best["sequence_id"]),
        "target_id": int(early_best["target_id"]),
        "gallery_frame": int(early_best["frame_index"]),
        "query_frame": int(late_best["frame_index"]),
        "frame_gap": frame_gap,
        "gallery_row": early_best.to_dict(),
        "query_row": late_best.to_dict()
    })

identity_pairs = pd.DataFrame(pair_records)

if identity_pairs.empty:
    raise RuntimeError(
        "No identities have valid earlier/later observations with the "
        "configured frame gap. Inspect the sequence annotations and consider "
        "reducing MIN_QUERY_GALLERY_GAP_FRAMES or relaxing crop filters."
    )

print("Eligible sequence-local identities:", len(identity_pairs))
print("Sequences represented:", identity_pairs["sequence_id"].nunique())
print("Median temporal gap (frames):", identity_pairs["frame_gap"].median())

display(
    identity_pairs[
        [
            "sequence_id", "target_id", "gallery_frame",
            "query_frame", "frame_gap"
        ]
    ].head(20)
)

## Section 9 — Extract the ground-truth crops

Use the supplied frame index to locate the correct JPG frame. Use the ground-truth bounding box to crop the person.

Each identity receives:
- One `gallery` crop from the earlier part of its sequence.
- One `query` crop from the later part of its sequence.

The resulting crop manifest keeps the source sequence and ground-truth target ID for evaluation.

In [ ]:
crop_rows = []

def extract_ground_truth_crop(annotation_row):
    sequence_name = str(annotation_row["sequence_id"])
    frame_number = int(annotation_row["frame_index"])

    frame_map = sequence_data[sequence_name]["frame_map"]

    if frame_number not in frame_map:
        return None

    image_path = frame_map[frame_number]
    image = cv2.imread(str(image_path))

    if image is None:
        return None

    x = float(annotation_row["bbox_left"])
    y = float(annotation_row["bbox_top"])
    w = float(annotation_row["bbox_width"])
    h = float(annotation_row["bbox_height"])

    margin_x = int(w * CROP_MARGIN_RATIO)
    margin_y = int(h * CROP_MARGIN_RATIO)

    left = max(0, int(math.floor(x)) - margin_x)
    top = max(0, int(math.floor(y)) - margin_y)
    right = min(image.shape[1], int(math.ceil(x + w)) + margin_x)
    bottom = min(image.shape[0], int(math.ceil(y + h)) + margin_y)

    crop = image[top:bottom, left:right]

    if crop.size == 0:
        return None

    return crop, image_path


for _, pair in identity_pairs.iterrows():
    for role in ["gallery", "query"]:
        annotation_row = pair[f"{role}_row"]
        extracted = extract_ground_truth_crop(annotation_row)

        if extracted is None:
            continue

        crop, source_image = extracted

        sequence_name = str(pair["sequence_id"])
        target_id = int(pair["target_id"])
        frame_number = int(annotation_row["frame_index"])

        output_folder = (
            CROP_DIR / sequence_name / f"target_{target_id:05d}"
        )
        output_folder.mkdir(parents=True, exist_ok=True)

        crop_path = output_folder / (
            f"{role}_frame_{frame_number:07d}.jpg"
        )

        if not cv2.imwrite(str(crop_path), crop):
            continue

        crop_height, crop_width = crop.shape[:2]

        crop_rows.append({
            "sequence_id": sequence_name,
            "target_id": target_id,
            "identity_key": f"{sequence_name}::target_{target_id}",
            "role": role,
            "frame_index": frame_number,
            "frame_gap": int(pair["frame_gap"]),
            "crop_width": crop_width,
            "crop_height": crop_height,
            "crop_path": str(crop_path),
            "source_image": str(source_image)
        })

crop_manifest = pd.DataFrame(crop_rows)

if crop_manifest.empty:
    raise RuntimeError("No ground-truth crops were extracted.")

CROP_MANIFEST_PATH = OUTPUT_DIR / "visdrone_reid_crop_manifest.csv"
crop_manifest.to_csv(CROP_MANIFEST_PATH, index=False)

print("Ground-truth crops saved:", len(crop_manifest))
print("Identities with crops:", crop_manifest["identity_key"].nunique())
print("Gallery crops:", int((crop_manifest["role"] == "gallery").sum()))
print("Query crops:", int((crop_manifest["role"] == "query").sum()))
print("Crop manifest:", CROP_MANIFEST_PATH)

display(crop_manifest.head(10))

## Section 10 — Inspect the ground-truth crop gallery

Check that gallery and query crops contain the correct people. Some crops may still be very small because the original aerial frames show distant people.

If the crop gallery mostly contains background, incorrect boxes or extremely tiny people, stop and share the screenshot before interpreting the ReID metrics.

In [ ]:
gallery_preview = crop_manifest.head(30).copy()

columns = 5
rows = math.ceil(len(gallery_preview) / columns)

fig, axes = plt.subplots(
    rows, columns,
    figsize=(columns * 3, rows * 3)
)
axes = np.array(axes).reshape(-1)

for ax in axes:
    ax.axis("off")

for index, row in gallery_preview.iterrows():
    image = cv2.imread(row["crop_path"])

    if image is None:
        continue

    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    axes[index].imshow(image)
    axes[index].set_title(
        f"{row['sequence_id']}\n"
        f"ID {row['target_id']} | {row['role']}\n"
        f"{row['crop_width']}x{row['crop_height']}"
    )
    axes[index].axis("off")

plt.tight_layout()
plt.show()

## Section 11 — Load pretrained OSNet and extract embeddings

We use the same pretrained OSNet x1.0 checkpoint as Part 3. The model returns one 512-dimensional feature vector for each crop.

This is feature extraction only. We are not fine-tuning OSNet in this notebook.

In [ ]:
reid_model = ReIDModel.from_pretrained(
    REID_MODEL_NAME,
    device="cuda" if torch.cuda.is_available() else "cpu"
)

crop_paths = crop_manifest["crop_path"].tolist()

embedding_start = time.time()

embeddings = reid_model.extract_embeddings_from_paths(
    crop_paths,
    batch_size=EMBEDDING_BATCH_SIZE,
    num_workers=0
)

embeddings = np.asarray(embeddings, dtype=np.float32)

if embeddings.ndim != 2:
    raise ValueError(f"Expected 2D embeddings, received {embeddings.shape}")

if embeddings.shape[0] != len(crop_manifest):
    raise ValueError("Embedding count does not match crop count.")

if not np.isfinite(embeddings).all():
    raise ValueError("Non-finite values found in the embedding matrix.")

# Normalize every feature vector for cosine similarity.
embedding_norms = np.linalg.norm(embeddings, axis=1, keepdims=True)
embeddings = embeddings / np.maximum(embedding_norms, 1e-12)

embedding_seconds = time.time() - embedding_start

print("Embedding matrix:", embeddings.shape)
print("Embedding dimensions:", embeddings.shape[1])
print("Embedding time (seconds):", round(embedding_seconds, 2))

## Section 12 — Evaluate Rank-1 accuracy and mAP

For each sequence separately:
- The gallery contains earlier crops.
- The query set contains later crops.
- The ground-truth identity key is `sequence_id::target_id`.
- A query is correct when its matching identity is ranked first.
- Average Precision (AP) is computed from the ranking of relevant gallery items.
- mAP is the mean AP over queries.

We deliberately evaluate each sequence separately. VisDrone target IDs are not global across sequences, so we must not compare them as if they were shared identities.

With one gallery crop per identity, each query has one relevant gallery image. This is a straightforward first retrieval benchmark, not a full multi-shot ReID protocol.

In [ ]:
eval_manifest = crop_manifest.reset_index(drop=True).copy()
eval_manifest["embedding_index"] = np.arange(len(eval_manifest))

gallery_metrics = []
query_metrics = []
pairwise_scores = []
sequence_metrics = []

for sequence_name in sorted(eval_manifest["sequence_id"].unique()):
    sequence_rows = eval_manifest[
        eval_manifest["sequence_id"] == sequence_name
    ]

    gallery_rows = sequence_rows[
        sequence_rows["role"] == "gallery"
    ].reset_index(drop=True)

    query_rows = sequence_rows[
        sequence_rows["role"] == "query"
    ].reset_index(drop=True)

    if len(gallery_rows) < 2 or len(query_rows) < 2:
        print(
            f"Skipping {sequence_name}: insufficient gallery/query identities."
        )
        continue

    gallery_vectors = embeddings[
        gallery_rows["embedding_index"].to_numpy()
    ]
    query_vectors = embeddings[
        query_rows["embedding_index"].to_numpy()
    ]

    # Cosine similarity because embeddings are L2-normalized.
    similarities = query_vectors @ gallery_vectors.T

    gallery_keys = gallery_rows["identity_key"].to_numpy()
    query_keys = query_rows["identity_key"].to_numpy()

    relevance = (
        query_keys[:, None] == gallery_keys[None, :]
    )

    sequence_aps = []
    sequence_rank1 = []

    for query_index, query_row in query_rows.iterrows():
        scores = similarities[query_index]
        relevant = relevance[query_index]

        ranking = np.argsort(scores)[::-1]
        ranked_relevance = relevant[ranking]

        relevant_positions = np.where(ranked_relevance)[0]

        if len(relevant_positions) == 0:
            continue

        first_rank = int(relevant_positions[0]) + 1
        rank1 = int(first_rank == 1)

        # Average precision for this query.
        cumulative_hits = np.cumsum(ranked_relevance)
        precisions_at_hits = (
            cumulative_hits[relevant_positions] /
            (relevant_positions + 1)
        )
        average_precision = float(
            precisions_at_hits.mean()
        )

        sequence_aps.append(average_precision)
        sequence_rank1.append(rank1)

        query_metrics.append({
            "sequence_id": sequence_name,
            "identity_key": query_row["identity_key"],
            "target_id": int(query_row["target_id"]),
            "rank_of_correct_identity": first_rank,
            "rank1_correct": rank1,
            "average_precision": average_precision,
            "best_similarity": float(scores[ranking[0]]),
            "correct_identity_similarity": float(
                scores[np.where(relevant)[0][0]]
            )
        })

        for gallery_index, gallery_row in gallery_rows.iterrows():
            pairwise_scores.append({
                "sequence_id": sequence_name,
                "query_identity": query_row["identity_key"],
                "gallery_identity": gallery_row["identity_key"],
                "same_identity": int(relevant[gallery_index]),
                "cosine_similarity": float(scores[gallery_index])
            })

    if sequence_aps:
        sequence_metrics.append({
            "sequence_id": sequence_name,
            "queries": len(sequence_aps),
            "gallery_identities": len(gallery_rows),
            "rank1_accuracy": float(np.mean(sequence_rank1)),
            "mAP": float(np.mean(sequence_aps))
        })

query_metrics_df = pd.DataFrame(query_metrics)
pairwise_scores_df = pd.DataFrame(pairwise_scores)
sequence_metrics_df = pd.DataFrame(sequence_metrics)

if query_metrics_df.empty:
    raise RuntimeError(
        "No queries could be evaluated. Check the split, crop filters "
        "and whether the sequences contain identities visible in both halves."
    )

overall_rank1 = float(query_metrics_df["rank1_correct"].mean())
overall_map = float(query_metrics_df["average_precision"].mean())

print("Overall Rank-1 accuracy:", round(overall_rank1, 4))
print("Overall mAP:", round(overall_map, 4))
print("Queries evaluated:", len(query_metrics_df))

display(sequence_metrics_df)

## Section 13 — Pairwise verification diagnostics

This section treats each query/gallery comparison as:
- Positive: same sequence-local ground-truth identity.
- Negative: different ground-truth identities within the same sequence.

We report ROC AUC and average precision over all pair comparisons. These are useful diagnostics, but they are affected by the large number of negative pairs.

We also show precision/recall at a few candidate thresholds. Do not choose a production threshold from these seven validation sequences alone.

In [ ]:
if pairwise_scores_df.empty:
    raise RuntimeError("No pairwise scores were produced.")

pair_labels = pairwise_scores_df["same_identity"].to_numpy()
pair_scores = pairwise_scores_df["cosine_similarity"].to_numpy()

if len(np.unique(pair_labels)) == 2:
    pair_roc_auc = float(roc_auc_score(pair_labels, pair_scores))
    pair_average_precision = float(
        average_precision_score(pair_labels, pair_scores)
    )
else:
    pair_roc_auc = np.nan
    pair_average_precision = np.nan

print("Pairwise ROC AUC:", pair_roc_auc)
print("Pairwise average precision:", pair_average_precision)

threshold_rows = []

for threshold in [0.50, 0.60, 0.70, 0.75, 0.80, 0.85, 0.90, 0.92]:
    predictions = (pair_scores >= threshold).astype(int)

    tp = int(((predictions == 1) & (pair_labels == 1)).sum())
    fp = int(((predictions == 1) & (pair_labels == 0)).sum())
    fn = int(((predictions == 0) & (pair_labels == 1)).sum())
    tn = int(((predictions == 0) & (pair_labels == 0)).sum())

    precision = tp / (tp + fp) if tp + fp else 0.0
    recall = tp / (tp + fn) if tp + fn else 0.0
    f1 = (
        2 * precision * recall / (precision + recall)
        if precision + recall else 0.0
    )

    threshold_rows.append({
        "threshold": threshold,
        "TP": tp,
        "FP": fp,
        "FN": fn,
        "TN": tn,
        "precision": precision,
        "recall": recall,
        "F1": f1
    })

threshold_df = pd.DataFrame(threshold_rows)
display(threshold_df)

## Section 14 — Review correct and incorrect retrieval examples

Inspect the highest-scoring incorrect matches and the lowest-scoring correct matches. This helps explain where OSNet succeeds and fails on aerial crops.

The first crop is the later query. The second crop is the top-ranked gallery result.

In [ ]:
# Determine the top gallery candidate for each evaluated query.
top_match_rows = []

for sequence_name in sorted(eval_manifest["sequence_id"].unique()):
    sequence_rows = eval_manifest[
        eval_manifest["sequence_id"] == sequence_name
    ]

    gallery_rows = sequence_rows[
        sequence_rows["role"] == "gallery"
    ].reset_index(drop=True)

    query_rows = sequence_rows[
        sequence_rows["role"] == "query"
    ].reset_index(drop=True)

    if gallery_rows.empty or query_rows.empty:
        continue

    gallery_vectors = embeddings[
        gallery_rows["embedding_index"].to_numpy()
    ]
    query_vectors = embeddings[
        query_rows["embedding_index"].to_numpy()
    ]

    similarities = query_vectors @ gallery_vectors.T

    for q_index, q_row in query_rows.iterrows():
        ranking = np.argsort(similarities[q_index])[::-1]
        best_gallery_index = int(ranking[0])
        best_gallery = gallery_rows.iloc[best_gallery_index]

        same_identity = (
            q_row["identity_key"] == best_gallery["identity_key"]
        )

        top_match_rows.append({
            "sequence_id": sequence_name,
            "query_identity": q_row["identity_key"],
            "gallery_identity": best_gallery["identity_key"],
            "same_identity": same_identity,
            "similarity": float(similarities[q_index, best_gallery_index]),
            "query_crop": q_row["crop_path"],
            "gallery_crop": best_gallery["crop_path"]
        })

top_matches_df = pd.DataFrame(top_match_rows)

false_matches = (
    top_matches_df[top_matches_df["same_identity"] == False]
    .sort_values("similarity", ascending=False)
    .head(8)
)

missed_matches = (
    top_matches_df[top_matches_df["same_identity"] == True]
    .sort_values("similarity", ascending=True)
    .head(8)
)

print("Highest-scoring incorrect top matches:")
display(false_matches[["sequence_id", "query_identity",
                       "gallery_identity", "similarity"]])

print("Lowest-scoring correct top matches:")
display(missed_matches[["sequence_id", "query_identity",
                        "gallery_identity", "similarity"]])

## Section 15 — Save evaluation results

The notebook saves:
- `visdrone_gt_person_annotations.csv`: filtered person ground truth.
- `visdrone_reid_crop_manifest.csv`: gallery/query crop locations and labels.
- `visdrone_osnet_embeddings.npz`: normalized 512-dimensional crop embeddings.
- `visdrone_reid_query_metrics.csv`: Rank-1 and AP for each query.
- `visdrone_reid_pairwise_scores.csv`: same/different pair labels and cosine scores.
- `visdrone_reid_sequence_metrics.csv`: per-sequence Rank-1 and mAP.
- `visdrone_reid_threshold_metrics.csv`: threshold diagnostic table.
- `visdrone_reid_top_matches.csv`: top retrieval result for each query.
- `visdrone_reid_summary.json`: overall experiment summary.

In [ ]:
# Save all evaluation artefacts.
person_annotations.to_csv(
    OUTPUT_DIR / "visdrone_gt_person_annotations.csv",
    index=False
)

crop_manifest.to_csv(
    OUTPUT_DIR / "visdrone_reid_crop_manifest.csv",
    index=False
)

np.savez_compressed(
    OUTPUT_DIR / "visdrone_osnet_embeddings.npz",
    embeddings=embeddings.astype(np.float32),
    identity_keys=crop_manifest["identity_key"].to_numpy(dtype=str),
    sequence_ids=crop_manifest["sequence_id"].to_numpy(dtype=str),
    roles=crop_manifest["role"].to_numpy(dtype=str)
)

query_metrics_df.to_csv(
    OUTPUT_DIR / "visdrone_reid_query_metrics.csv",
    index=False
)

pairwise_scores_df.to_csv(
    OUTPUT_DIR / "visdrone_reid_pairwise_scores.csv",
    index=False
)

sequence_metrics_df.to_csv(
    OUTPUT_DIR / "visdrone_reid_sequence_metrics.csv",
    index=False
)

threshold_df.to_csv(
    OUTPUT_DIR / "visdrone_reid_threshold_metrics.csv",
    index=False
)

top_matches_df.to_csv(
    OUTPUT_DIR / "visdrone_reid_top_matches.csv",
    index=False
)

summary = {
    "sequences_found": len(sequence_data),
    "valid_person_annotations": int(len(person_annotations)),
    "eligible_identities": int(len(identity_pairs)),
    "gallery_crops": int((crop_manifest["role"] == "gallery").sum()),
    "query_crops": int((crop_manifest["role"] == "query").sum()),
    "embedding_dimensions": int(embeddings.shape[1]),
    "queries_evaluated": int(len(query_metrics_df)),
    "rank1_accuracy": overall_rank1,
    "mAP": overall_map,
    "pairwise_roc_auc": pair_roc_auc,
    "pairwise_average_precision": pair_average_precision,
    "osnet_model": REID_MODEL_NAME,
    "note": (
        "Within-sequence temporal ReID evaluation using VisDrone ground truth. "
        "IDs are local to each sequence. Results do not establish cross-video "
        "identity matching."
    )
}

with open(
    OUTPUT_DIR / "visdrone_reid_summary.json",
    "w",
    encoding="utf-8"
) as file:
    json.dump(summary, file, indent=2)

print("Results saved to:", OUTPUT_DIR)
display(pd.DataFrame([summary]))

## Section 16 — How to interpret this experiment

### Rank-1 accuracy
The fraction of queries for which the correct gallery identity is the top-ranked result.

### mAP
Mean Average Precision across queries. With one gallery crop per identity, each query has one relevant gallery item, so AP is the reciprocal of the rank of the correct identity.

### Pairwise ROC AUC
Measures how well cosine scores rank same-identity pairs above different-identity pairs.

### Important limitations
- This validation set contains only seven sequences; results may vary substantially by sequence.
- We intentionally compare identities only within each sequence because target IDs are sequence-local.
- Our temporal split selects one crop per identity from the earlier half and one from the later half. This is a controlled baseline, not a full benchmark protocol.
- The model was pretrained on MSMT17, not trained on aerial imagery.
- The crop filters remove heavily occluded and very small objects, so the results do not describe performance on every person in VisDrone.
- Strong performance here would support further experiments, but it would not prove that OSNet can identify the same person across unrelated videos.

### Next step
Compare the per-sequence results and inspect the false matches. If performance is weak, we can test an aerial-specific ReID model or fine-tune OSNet using a separate training split. Do not fine-tune on this validation subset and then report its performance on the same identities as an independent test.